In [ ]:
import pandas as pd
import numpy as np
import zipfile

with zipfile.ZipFile('dialectro.zip') as zip_ref:
    zip_ref.extractall()

#Load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df.head()

# Subtasks 1, 2 & 3

In [ ]:
#Subtask 1
import re

answer_sub1 = 0
for df in [train_df, test_df]:
    num_aparitii = df['text'].str.lower().apply(lambda x: len(re.findall(r'\bpâni\b', x)))
    answer_sub1 += num_aparitii.sum()

print(answer_sub1)

In [ ]:
#Subtask 2
import string

sub2_function = lambda text: len([c for c in text if c in string.punctuation])

mold = train_df.loc[train_df['label'] == 'graiul moldovenesc', 'text'].apply(sub2_function).mean()
banat = train_df.loc[train_df['label'] == 'graiul bănățean', 'text'].apply(sub2_function).mean()

answer_sub2 = round(abs(banat-mold),2)

print(answer_sub2)

In [ ]:
#Subtask 3
def count_diacritice(text):
    diacritice = 'ăâîșț'
    num_diacritice = 0
    for c in text.lower():
        if c in diacritice:
            num_diacritice += 1
    return num_diacritice

answer_sub3 = list(test_df['text'].apply(count_diacritice))
print(answer_sub3[:10])

# Subtask 4

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device', device)

#create a dataset and a loader for efficiency
class TextDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        return self.df.iloc[index]['text']

train_loader = DataLoader(TextDataset(train_df), batch_size=32, shuffle=False)
test_loader = DataLoader(TextDataset(test_df), batch_size=32, shuffle=False)

In [ ]:
from transformers import AutoTokenizer, AutoModel

def mean_pooling(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def get_embeddings(loader, model_name='distilbert-base-multilingual-cased', device=device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(device)
    model.eval()

    embeddings = []

    with torch.no_grad():
        for texts in loader:
            texts = tokenizer(texts, return_tensors='pt', truncation=True, padding=True, max_length=512)
            texts = {k: v.to(device) for k, v in texts.items()}

            batch_emb = model(**texts)
            batch_emb = mean_pooling(batch_emb.last_hidden_state, texts['attention_mask'])

            embeddings.extend(batch_emb.cpu().numpy())

    return np.array(embeddings)

X_train = get_embeddings(train_loader)
X_test = get_embeddings(test_loader)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(train_df['label'])

model = MLPClassifier(
    max_iter=750,
    hidden_layer_sizes=(128, 64),
    random_state=42
)
model.fit(X_train, y_train)

preds = le.inverse_transform(model.predict(X_test))
print(preds[:5])

In [ ]:
#Submission
output_df = pd.DataFrame({
    'subtaskID':[1, 2] + [3] *len(test_df) + [4] * len(test_df),
    'datapointID':[1,1] + list(test_df['ID']) * 2,
    'answer':[answer_sub1, answer_sub2] + answer_sub3 + list(preds)
})

output_df.to_csv('submission.csv', index=False)
output_df.head()

100p/100p; F1: 1